# R Tutorial: Plot A Tiny Expression Matrix

**End goal:** join gene counts with sample metadata and plot expression for a marker gene.

This notebook uses base R so it works in the Week 1 starter environment without extra packages.

## Setup: load the tutorial data

Run this cell first. It uses the local files when they already exist and downloads the tiny tutorial data when you open the notebook in Colab or another fresh environment.

In [ ]:
data_dir <- '../data'
dir.create(data_dir, showWarnings = FALSE)
raw_base <- 'https://raw.githubusercontent.com/Caffeinated-Code/Bioinformatics-Field-Guide/main/content/resources/week-02/data'
files <- c('samples.tsv', 'mini_transcripts.fasta', 'gene_counts.tsv', 'gene_annotations.tsv')
for (file_name in files) {
  target <- file.path(data_dir, file_name)
  if (!file.exists(target)) {
    download.file(paste0(raw_base, '/', file_name), target, quiet = TRUE)
  }
}
list.files(data_dir)


In [ ]:
samples <- read.delim('../data/samples.tsv')
counts <- read.delim('../data/gene_counts.tsv', check.names = FALSE)
annotations <- read.delim('../data/gene_annotations.tsv')

samples
counts

## 1. Choose a gene and reshape counts

Many R workflows begin by reshaping data into a form that is easy to model or plot.

In [ ]:
gene_id <- 'ENSG00000141510'
gene_row <- counts[counts$gene_id == gene_id, ]

long_counts <- data.frame(
  gene_id = gene_id,
  sample_id = names(gene_row)[-1],
  count = as.numeric(gene_row[1, -1])
)

long_counts

## 2. Join sample metadata

Never interpret counts without knowing which sample is which.

In [ ]:
plot_data <- merge(long_counts, samples, by = 'sample_id')
plot_data <- merge(plot_data, annotations, by = 'gene_id', all.x = TRUE)
plot_data

## 3. Plot expression by condition

This is not a differential expression test. It is a quick visual check.

In [ ]:
boxplot(
  count ~ condition,
  data = plot_data,
  main = paste('Counts for', gene_id),
  xlab = 'Condition',
  ylab = 'Raw counts',
  col = c('#8fb3ff', '#f2a65a')
)
stripchart(count ~ condition, data = plot_data, vertical = TRUE, method = 'jitter', add = TRUE, pch = 19)

## 4. Compare group means

Use this only as a descriptive check. Real RNA-seq differential expression needs models such as DESeq2, edgeR, or limma-voom.

In [ ]:
aggregate(count ~ condition, data = plot_data, FUN = mean)

## Your turn

- Change `gene_id` to `ENSG00000157764` and rerun the notebook.
- Change the plot title to use gene symbol instead of Ensembl ID.
- Ask whether batch might matter for this tiny example.

**Confidence check:** you should know why this plot is useful, and why it is not a statistical test.